# CS50 Lecture 5 Key Notes

## 5.1 Abstract Data Types

Abstract data types are conceptual designs for data structures. They define the characteristics, features, and operations a structure must offer, independent of how it is implemented in low-level code.

### 5.1.1 Queues

A **queue** is a data structure that exhibits a **First In, First Out (FIFO)** property. The first element added to the queue is the first element removed. This property models fairness, similar to a line of customers waiting at a store. 

A queue supports two primary operations:
* **Enqueue**: Adding an element to the back of the line.
* **Dequeue**: Removing an element from the front of the line.

A basic queue can be implemented in C using an array and a structure that tracks the array's capacity and current size. However, statically allocated arrays introduce rigid limitations. If a queue is defined with a capacity of 50, a 51st element cannot be added without recompiling the program with a larger array. Conversely, allocating an excessively large array wastes memory if only a few elements are enqueued.

### 5.1.2 Stacks

A **stack** is a data structure that exhibits a **Last In, First Out (LIFO)** property. The most recently added element is the first one removed. This behaves like a stack of cafeteria trays or a pile of clothes, where items are always taken from the very top. Another real-world analog is an email inbox, where the newest messages appear at the top and are typically addressed first.

A stack supports two primary operations:
* **Push**: Adding an element to the top of the stack.
* **Pop**: Removing an element from the top of the stack.

Like queues, a stack can be implemented using a fixed-size array. Elements are added and removed from the end of the array, leaving the oldest element securely at index `0`. This implementation shares the same limitation as the array-based queue: it cannot dynamically grow or shrink to accommodate changing memory needs.

### 5.1.3 Dictionaries

A **dictionary** is an abstract data type that maps a **key** to a **value**. This forms a two-column association, identical to a physical dictionary linking words to definitions or a phone book linking names to numbers. Dictionaries are used extensively when a piece of data must be associated with another piece of data. 

Like stacks and queues, dictionaries can be implemented using arrays, but this approach introduces constraints on capacity and performance. Different underlying implementations dictate how quickly a dictionary can be searched or updated.

### 5.1.4 Data Structures Cheat Sheet

| Data Structure | Property | Core Operations | Real-World Analogy |
|---|---|---|---|
| **Queue** | FIFO (First In, First Out) | Enqueue, Dequeue | A checkout line at a store |
| **Stack** | LIFO (Last In, First Out) | Push, Pop | A stack of cafeteria trays |
| **Dictionary** | Key-Value Pairs | Insert, Lookup | A phone book |

### 5.1.5 Questions

1. What is the fundamental difference between the FIFO property of a queue and the LIFO property of a stack?
2. If a program requires fairness where the oldest pending task is handled first, should a stack or a queue be used?
3. What is the primary limitation of implementing an abstract data type using a statically allocated array?

## 5.2 Resizing Arrays

Arrays allocate a contiguous chunk of memory for values to be stored back-to-back. Because an array's size is generally fixed at compilation time, running out of space requires manually recompiling the code with a larger size or writing logic to dynamically request more memory from the operating system while the program runs.

### 5.2.1 Dynamically Allocating an Array

Instead of statically defining an array (e.g., `int list[3];`), an array can be created dynamically using `malloc`. This approach asks the operating system for a specific number of bytes at runtime.

```c
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    // Dynamically allocate space for 3 integers
    int *list = malloc(3 * sizeof(int));
    if (list == NULL)
    {
        return 1;
    }

    // Populate the array
    list[0] = 1;
    list[1] = 2;
    list[2] = 3;

    // Print the array
    for (int i = 0; i < 3; i++)
    {
        printf("%i\n", list[i]);
    }
    
    free(list);
    return 0;
}
```

Tracing this program:
1. `malloc(3 * sizeof(int))` requests 12 bytes of contiguous memory (assuming a 4-byte integer).
2. `malloc` returns the address of the first byte of this chunk, which is stored in the pointer `list`.
3. `if (list == NULL)` acts as a safeguard. If the operating system cannot provide the memory, `malloc` returns `NULL`, and the program safely exits.
4. The bracket notation `list[0]` is syntactic sugar for pointer arithmetic (`*list`). The values `1`, `2`, and `3` are stored contiguously in the allocated block.
5. `free(list)` returns the memory to the operating system before the program concludes.

### 5.2.2 Growing an Array with malloc

If an array must grow to accommodate a fourth element, the program cannot simply append the value to the end of the existing memory block. The neighboring bytes might belong to another variable, such as a string, and overwriting them would corrupt data. 

To safely grow an array, a completely new, larger chunk of memory must be allocated. The old values are copied over, the new value is appended, and the old memory is freed.

```c
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    // Allocate list of size 3
    int *list = malloc(3 * sizeof(int));
    if (list == NULL)
    {
        return 1;
    }

    list[0] = 1;
    list[1] = 2;
    list[2] = 3;

    // Allocate new list of size 4
    int *tmp = malloc(4 * sizeof(int));
    if (tmp == NULL)
    {
        free(list);
        return 1;
    }

    // Copy list of size 3 into list of size 4
    for (int i = 0; i < 3; i++)
    {
        tmp[i] = list[i];
    }

    // Add number to list of size 4
    tmp[3] = 4;

    // Free list of size 3
    free(list);

    // Remember list of size 4
    list = tmp;

    // Print list
    for (int i = 0; i < 4; i++)
    {
        printf("%i\n", list[i]);
    }

    // Free list
    free(list);
    return 0;
}
```

Tracing the resizing logic:
1. `int *tmp = malloc(4 * sizeof(int));` requests 16 bytes for the new array. A temporary variable (`tmp`) must be used. If the new pointer were directly assigned to `list`, and `malloc` failed (returning `NULL`), the address of the original 12 bytes would be permanently lost, causing a memory leak.
2. `if (tmp == NULL)` checks for allocation failure. If it fails, the original `list` is explicitly freed before returning `1`. 
3. The `for` loop iterates through the original three elements, copying them to the exact same indices in `tmp`.
4. `tmp[3] = 4;` appends the new value to the newly secured fourth slot.
5. `free(list);` releases the original 12 bytes back to the operating system.
6. `list = tmp;` updates the `list` pointer to point to the new 16-byte block. The temporary pointer is no longer needed.

### 5.2.3 Growing an Array with realloc

Manually allocating new memory, copying elements via a loop, and freeing the old memory is tedious. The `realloc` function automates this entire process.

```c
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    // Allocate list of size 3
    int *list = malloc(3 * sizeof(int));
    if (list == NULL)
    {
        return 1;
    }

    list[0] = 1;
    list[1] = 2;
    list[2] = 3;

    // Resize list to be of size 4
    int *tmp = realloc(list, 4 * sizeof(int));
    if (tmp == NULL)
    {
        free(list);
        return 1;
    }
    
    list = tmp;

    // Add number to list
    list[3] = 4;

    // Print list
    for (int i = 0; i < 4; i++)
    {
        printf("%i\n", list[i]);
    }

    free(list);
    return 0;
}
```

`realloc` accepts two arguments: the address of the existing memory chunk (`list`), and the new desired total size in bytes (`4 * sizeof(int)`). 

Tracing `realloc`:
1. `realloc` checks if the memory immediately following the existing `list` array is vacant. 
2. If vacant, `realloc` simply claims the adjacent bytes and returns the original address. This avoids moving any data.
3. If occupied, `realloc` automatically allocates a new chunk of memory elsewhere, copies the existing data over, and frees the original chunk. It then returns the address of the new chunk.
4. Regardless of the outcome, `realloc` returns `NULL` on failure, meaning the temporary variable `tmp` remains strictly necessary to prevent memory leaks.

### 5.2.4 Questions

1. Why is a temporary pointer (`tmp`) strictly required when resizing an array with `malloc` or `realloc`?
2. If `realloc` is asked to resize an array, under what condition does it avoid moving the array to a new location in memory?
3. What happens if a programmer attempts to append a fourth element to an array statically sized for three elements (e.g. `list[3] = 4;`) without resizing it?

## 5.3 Linked Lists

A **linked list** solves the array's contiguous memory problem by stitching together discrete chunks of memory. Instead of requiring memory to be back-to-back, a linked list allows elements to be stored anywhere in memory. This is achieved by storing **metadata** alongside the data itself.

In a linked list, each element is a **node**. A node is a custom data structure containing two components:
1. The **data** (e.g., an integer).
2. A **pointer** to the next node in the list.

The end of the list is marked by setting the final pointer to `NULL`. The entire list is kept track of using a single pointer (often called the "head" or simply `list`), which holds the address of the very first node.

### 5.3.1 Defining a Node in C

A node can be defined using a `struct`.

```c
typedef struct node
{
    int number;
    struct node *next;
} node;
```

Notice the line `struct node *next;`. The structure must be temporarily named `struct node` at the very top (before the final `typedef` name) because the compiler reads top-to-bottom. When it reaches the declaration of the `next` pointer, it must already know what a `node` is.

### 5.3.2 Building a Linked List (Prepending)

To construct a linked list, space must be individually allocated for each node.

```c
#include <stdio.h>
#include <stdlib.h>

typedef struct node
{
    int number;
    struct node *next;
} node;

int main(void)
{
    // Initialize an empty list
    node *list = NULL;

    // Allocate memory for a new node
    node *n = malloc(sizeof(node));
    if (n == NULL)
    {
        return 1;
    }

    // Populate the node
    n->number = 1;
    n->next = NULL;

    // Prepend node to the list
    n->next = list;
    list = n;

    return 0;
}
```

The syntax `n->number` is shorthand (syntactic sugar) for `(*n).number`. It means: "Go to the address in `n` and access the `number` field."

Tracing the prepending logic:
1. `node *list = NULL;` initializes an empty list. The list pointer exists, but it points to nothing.
2. `node *n = malloc(sizeof(node));` asks the operating system for a chunk of memory large enough for a single node, storing its address in the temporary pointer `n`.
3. `n->number = 1;` populates the data field.
4. `n->next = NULL;` safely initializes the pointer field.
5. `n->next = list;` sets the new node's pointer to point to the current start of the list. (If the list is empty, it correctly points to `NULL`).
6. `list = n;` updates the list's main pointer to point to the newly created node, officially placing it at the front of the list.

> [!CAUTION]
> The order of the final two steps is critical. If `list = n;` is executed *before* `n->next = list;`, the main pointer immediately overwrites its only record of the existing list. The newly inserted node's `next` field will point nowhere, and the rest of the list becomes orphaned, causing an unrecoverable memory leak.

### 5.3.3 Traversing a Linked List

Unlike an array where pointer arithmetic (e.g., `list[i]`) can instantly calculate the address of any element, a linked list must be traversed linearly by following the pointers one by one.

```c
// Using a while loop
node *ptr = list;
while (ptr != NULL)
{
    printf("%i\n", ptr->number);
    ptr = ptr->next;
}
```

```c
// Using a for loop (equivalent logic)
for (node *ptr = list; ptr != NULL; ptr = ptr->next)
{
    printf("%i\n", ptr->number);
}
```

Tracing the traversal:
1. `node *ptr = list;` creates a temporary pointer `ptr` and sets it to the start of the list. It acts as a cursor moving through the data.
2. The loop condition `ptr != NULL` ensures traversal stops exactly at the end of the list.
3. `ptr->number` accesses the data for printing.
4. `ptr = ptr->next;` is the critical line. It updates the cursor `ptr` to hold the address of the *next* node, effectively moving the cursor forward one step.

### 5.3.4 Performance Analysis of Linked Lists

The design of a linked list yields significant performance trade-offs compared to an array. 

If elements are constantly **prepended** to the list (inserted at the very beginning), the running time for insertion is **O(1)**. The number of steps is strictly constant: allocate a node, point it to the list, update the list pointer. It takes the same amount of time regardless of whether the list has 0 or 1,000,000 elements.

However, because the values are scattered across memory and not placed in sorted order, searching and deleting are inefficient.
* **Search**: **O(N)**. Binary search is impossible because the middle element cannot be instantly located. The list must be checked sequentially from the beginning.
* **Deletion**: **O(N)**. The element must first be found before it can be removed.
* **Appending (Insertion at the end)**: **O(N)**. Unless a separate pointer tracks the end of the list, the entire list must be traversed to append a new node.

### 5.3.5 Questions

1. When defining the `node` structure, why is `struct node *next;` used instead of simply `node *next;`?
2. Explain why the `->` operator is necessary when working with a pointer to a struct, as opposed to the `.` operator.
3. If prepending a node takes O(1) time, why does traversing the list to append a node take O(N) time?

### 5.3.6 Inserting in Sorted Order

To maintain a sorted linked list as new nodes are added, every possible insertion scenario must be explicitly handled. The logic breaks down into four cases:
1. The list is empty.
2. The new number belongs at the very beginning (prepending).
3. The new number belongs at the very end (appending).
4. The new number belongs somewhere in the middle.

```c
// If list is empty
if (list == NULL)
{
    list = n;
}
// If number belongs at beginning of list
else if (n->number < list->number)
{
    n->next = list;
    list = n;
}
// If number belongs later in list
else
{
    // Iterate over nodes in list
    for (node *ptr = list; ptr != NULL; ptr = ptr->next)
    {
        // If at end of list
        if (ptr->next == NULL)
        {
            // Append node
            ptr->next = n;
            break;
        }
        // If in middle of list
        if (n->number < ptr->next->number)
        {
            n->next = ptr->next;
            ptr->next = n;
            break;
        }
    }
}
```

Tracing a middle insertion (e.g. inserting `3` between `2` and `4`):
1. The `for` loop cursor `ptr` points to the node `2`.
2. `ptr->next->number` looks ahead to the next node's value (`4`).
3. Because `3 < 4`, the correct position is found.
4. `n->next = ptr->next;` points the new node's arrow to `4`.
5. `ptr->next = n;` points `2`'s arrow to the new node (`3`).

### 5.3.7 Freeing a Linked List

When a linked list is no longer needed, every single node that was allocated using `malloc` must be returned to the system using `free`. The function cannot simply free the main `list` pointer because `free` has no understanding of custom data structures—it would only free the first node, leaving the rest stranded in memory.

```c
void unload(node *list)
{
    node *ptr = list;
    while (ptr != NULL)
    {
        node *next = ptr->next;
        free(ptr);
        ptr = next;
    }
}
```

Tracing the `unload` function:
1. `node *ptr = list;` sets the cursor to the first node.
2. The loop condition checks if the end has been reached.
3. `node *next = ptr->next;` saves the address of the *next* node. This step is mandatory. If the current node is freed first, its `next` pointer is destroyed, making it impossible to continue traversing.
4. `free(ptr);` safely deallocates the current node.
5. `ptr = next;` moves the cursor forward to the saved address to continue the process.

## 5.4 Trees

A binary search tree (BST) addresses the performance flaws of a linked list while retaining its dynamic memory allocation. By adding a second dimension, a BST regains the ability to perform binary search.

In a BST, each node contains data and *two* pointers: one for a "left child" and one for a "right child".

```c
typedef struct node
{
    int number;
    struct node *left;
    struct node *right;
} node;
```

A tree starts at a single **root** node and branches downward. The BST is defined by a strictly recursive property: for any given node, all elements in its left subtree must be smaller, and all elements in its right subtree must be larger.

### 5.4.1 Searching a Binary Search Tree

Because of the recursive property, searching a BST perfectly mimics the divide-and-conquer strategy of binary search.

```c
bool search(node *tree, int number)
{
    // Base case: empty tree (or reached the bottom without finding it)
    if (tree == NULL)
    {
        return false;
    }
    // Search left subtree
    else if (number < tree->number)
    {
        return search(tree->left, number);
    }
    // Search right subtree
    else if (number > tree->number)
    {
        return search(tree->right, number);
    }
    // Found the number
    else
    {
        return true;
    }
}
```

Tracing the search function:
1. `if (tree == NULL)` checks the base case. If the pointer handed to the function points nowhere, the number cannot possibly exist here.
2. `number < tree->number` determines if the target is smaller than the current node. If so, the entire right side of the tree is instantly eliminated, halving the problem. The function calls itself, passing only the left child.
3. `number > tree->number` does the inverse, diving down the right branch.
4. If it's not less than, and not greater than, it must be equal. `return true;`.

### 5.4.2 Performance Analysis of BSTs

The running time for operations on a balanced binary search tree is **O(log N)**.
* **Search**: The height of a balanced tree is $\log_2(N)$. Each step halves the remaining nodes.
* **Insertion**: Finding the correct spot to add a new node uses the same divide-and-conquer path.
* **Trade-off**: The primary cost is memory. Each node now requires space for an integer and two pointers (e.g., 20 bytes total instead of 4), consuming significantly more space than a standard array. Furthermore, if data is inserted in sorted order (e.g., 1, 2, 3), the tree degenerates into a long, stringy line, effectively becoming a linked list and devolving back to **O(N)** performance. Advanced algorithms exist to automatically "rebalance" the tree during insertion to prevent this.

### 5.4.3 Questions

1. When freeing a linked list, why must a temporary variable be used to store the `next` pointer before calling `free`?
2. How does a binary search tree utilize the concept of recursion in its structure?
3. What is the worst-case scenario for inserting elements into a binary search tree, and what is the resulting big O notation for searching that specific tree?

## 5.5 Hash Tables

While O(log N) is fast, the holy grail of algorithms is **O(1)**—constant time. A constant-time algorithm takes the exact same number of steps regardless of whether the dataset contains 10 items or 10 million items. To approach this, computer scientists mash together the fast indexing of an array with the dynamic memory of a linked list to create a **hash table**.

A hash table maps an infinite domain of possible values to a finite range of buckets. It is fundamentally an array of pointers to linked lists.

### 5.5.1 The Hash Function

The mechanism that determines which bucket a piece of data belongs in is the **hash function**. A hash function takes data (like a string) as input and returns a specific integer representing the bucket index.

```c
#include <ctype.h>

unsigned int hash(const char *name)
{
    return toupper(name[0]) - 'A';
}
```

Tracing the hash function:
1. `toupper(name[0])` grabs the first character of the string and ensures it is capitalized to standardize the math.
2. `- 'A'` (which is `- 65` in ASCII) converts the letter to a 0-indexed integer. 'A' becomes 0, 'B' becomes 1, up to 'Z' becoming 25.
3. The function returns this `unsigned int` to specify exactly which array bucket the name should be stored in. The `const` keyword in the parameter guarantees the function will not accidentally alter the original string in memory.

### 5.5.2 Resolving Collisions

Because the domain of names is infinite and the range of buckets (26) is finite, multiple names will inevitably hash to the same bucket (e.g., Mario and Master Chief both output 12). This is called a **collision**. 

To handle this without clobbering data, each bucket in the array does not store the data directly. Instead, the array elements are pointers to the head of a linked list. When a collision occurs, the new node is simply prepended to the linked list residing in that specific bucket.

This structure allows the program to jump instantly to the 'M' bucket in O(1) time. However, once inside the bucket, the program must traverse the linked list to find the specific 'M' name. If the hash table is poorly designed (e.g., only 26 buckets for a million names), the linked lists become extremely long. The running time mathematically devolves to **O(N)**. 

To mitigate this, a more complex hash function can be used (e.g., looking at the first three letters of a name). This creates vastly more buckets, reducing collisions and drastically speeding up the search, but at the cost of consuming a massive amount of memory.

### 5.5.3 Questions

1. Explain what a collision is in a hash table and how an array of linked lists solves the problem.
2. Why is the worst-case search time for a hash table O(N)?

## 5.6 Tries

A **trie** (short for retrieval) is a specialized tree composed entirely of arrays, designed to achieve true constant-time lookup for strings.

In a trie used for the English alphabet, every single node is a 26-element array of pointers.

```c
typedef struct node
{
    bool is_word;
    struct node *children[26];
} node;
```

To insert the word "TOAD":
1. Hash the first letter 'T' to access index 19 in the root array.
2. Allocate a new node and point index 19 to it.
3. In this new node, hash 'O' and point that index to a third node.
4. Repeat for 'A' and 'D'.
5. In the final node, flip the `is_word` boolean to `true`.

The word itself is never actually saved as a string; it is stored implicitly by the path taken through the pointers. 

Searching for "TOAD" simply involves following the arrays letter-by-letter and checking if the final boolean is true. The lookup time is bounded entirely by the length of the string, completely independent of how many millions of words are stored in the trie. It achieves **O(1)** constant-time lookup.

The fatal flaw of a trie is its memory consumption. Every node requires an array of 26 pointers, the vast majority of which remain `NULL` and unused. While it is incredibly fast, it wastes an enormous amount of space.

### 5.6.1 Questions

1. How does a trie achieve O(1) constant-time lookup for a word regardless of the dataset size?

## 5.7 Answers

### 5.1 Answers

**1.** A queue processes elements in the exact order they arrive (First In, First Out), much like a checkout line. A stack processes the most recently added element first (Last In, First Out), similar to taking the top tray from a cafeteria pile.

**2.** A queue must be used. A queue guarantees that the first task enqueued is the first task dequeued, preserving chronological fairness. A stack would process the newest tasks first, perpetually delaying older tasks.

**3.** The primary limitation is rigidity. A statically allocated array requires its size to be permanently fixed when the code is compiled. The abstract data type cannot dynamically grow if more space is needed, nor can it shrink to free up wasted memory if elements are removed.

### 5.2 Answers

**1.** If `malloc` or `realloc` fails to find enough memory, it returns `NULL`. If the program directly assigns this `NULL` result to the main `list` pointer (e.g., `list = realloc(list, size)`), the program instantaneously overwrites its only record of the original memory address. The original data becomes inaccessible, resulting in an unrecoverable memory leak. A temporary pointer secures the new address so failure can be safely checked before updating the main pointer.

**2.** `realloc` checks if the contiguous bytes immediately succeeding the original array are vacant. If they are completely empty and unallocated, `realloc` simply claims that adjacent space to meet the new size requirement, leaving the existing elements exactly where they are.

**3.** The program writes the value `4` into the memory space physically adjacent to the array's boundary. This memory is not owned by the array and might contain another variable's data, or it might be restricted by the operating system. Overwriting it leads to data corruption, unpredictable behavior, or an immediate segmentation fault.

### 5.3 Answers

**1.** When the compiler reads the `typedef struct` block top-to-bottom, the alias `node` has not been fully defined yet. Because the `next` pointer must reference the very structure currently being defined, it must use the temporary, formal name `struct node`. 

**2.** The `.` operator accesses a field of an actual structure in memory. When working with a pointer to a struct, the program does not have the struct itself; it has the address of the struct. The pointer must first be dereferenced (`*n`) before the `.` operator can be used (e.g., `(*n).number`). The `->` operator is syntactic sugar that automatically dereferences the pointer and accesses the field in a single step.

**3.** Prepending always occurs at the exact same location: the `list` head pointer. It requires only three operations regardless of the list's size. Appending requires attaching a node to the final element. Because a basic linked list only tracks the very first element, the program must jump from node to node, reading every single element until it finds the `NULL` terminator. For an `N`-element list, this requires `N` jumps.

### 5.4 Answers

**1.** If `free(ptr)` is executed immediately, the entire chunk of memory comprising that node is released to the operating system. This permanently destroys the `ptr->next` field. Without the address of the next node, the program has no way to find the rest of the linked list, resulting in a memory leak.

**2.** A binary search tree is recursive because every individual node within it obeys the exact same structural rules as the root node. The left child of any node is itself the root of a smaller, entirely valid binary search tree, and the right child is the root of another. This allows the exact same `search` function to be called repeatedly on progressively smaller subtrees.

**3.** The worst-case scenario occurs when elements are inserted in perfectly sorted order (e.g., 1, 2, 3, 4, 5). Because each subsequent number is larger, it always becomes the right child of the previous number. The tree degenerates into a single straight line, mathematically identical to a linked list. As a result, the search time devolves from O(log N) to **O(N)**.

### 5.5 Answers

**1.** A collision occurs when two distinct inputs (like "Mario" and "Master Chief") are mapped by the hash function to the exact same numerical output bucket. To solve this without deleting data, each bucket does not store the data itself, but rather a pointer to a linked list. When a collision happens, the new entry is safely prepended to the linked list residing in that specific bucket.

**2.** The worst-case search time occurs if every single piece of data fed into the hash table hashes to the exact same bucket. This forces every entry into a single, massive linked list. Searching for an item then requires a linear traversal of that linked list, resulting in O(N) performance.

### 5.6 Answers

**1.** In a trie, the data string itself acts as a map to navigate the array structure. To find a 4-letter word, the program makes exactly 4 array jumps, looking at the 1st letter, then the 2nd, the 3rd, and the 4th, before checking a boolean. Because it never compares the target word against other words in the trie, the lookup takes precisely 4 steps whether the trie contains 10 words or 10 million words. The speed is constrained only by the length of the string, which is considered a constant.